In [8]:
import os
import numpy as np

def load_bonn_dataset(path):

    X = []
    y = []

    folder_labels = {
        "A_Z":0,   # healthy
        "B_O":0,
        "C_N":1,   # epileptic region
        "D_F":1,
        "E_S":2    # seizure
    }

    for folder,label in folder_labels.items():

        folder_path = os.path.join(path, folder)

        for file in os.listdir(folder_path):

            if file.endswith(".txt"):

                signal = np.loadtxt(os.path.join(folder_path,file))

                X.append(signal)
                y.append(label)

    return np.array(X), np.array(y)


X,y = load_bonn_dataset(r"E:\Downloads\BONN dataset")

print("Signals:",X.shape)

Signals: (400, 4097)


In [9]:
import antropy as ant
from scipy.stats import skew, kurtosis

def hjorth(signal):

    diff1 = np.diff(signal)
    diff2 = np.diff(diff1)

    var0 = np.var(signal)
    var1 = np.var(diff1)
    var2 = np.var(diff2)

    mobility = np.sqrt(var1/var0)
    complexity = np.sqrt(var2/var1)/mobility

    return [var0, mobility, complexity]


def extract_features(signal):

    features = []

    # statistical
    features.append(np.mean(signal))
    features.append(np.std(signal))
    features.append(np.var(signal))
    features.append(np.max(signal))
    features.append(np.min(signal))
    features.append(np.ptp(signal))

    # Hjorth
    features.extend(hjorth(signal))

    # entropy
    features.append(ant.sample_entropy(signal))
    features.append(ant.app_entropy(signal))
    features.append(ant.perm_entropy(signal, normalize=True))
    features.append(ant.spectral_entropy(signal, sf=173, method='fft'))

     # NEW features
    features.append(skew(signal))
    features.append(kurtosis(signal))

    return features


X_features = np.array([extract_features(s) for s in X])

print("Feature shape:",X_features.shape)

Feature shape: (400, 15)


In [10]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X_features,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [12]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif

selector = SelectKBest(mutual_info_classif, k=15)

X_train = selector.fit_transform(X_train, y_train)
X_test = selector.transform(X_test)

In [13]:
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


estimators = [

 ('rf', RandomForestClassifier(
     n_estimators=500,
     max_depth=None
 )),

 ('xgb', XGBClassifier(
     n_estimators=400,
     learning_rate=0.05,
     max_depth=5,
     subsample=0.9,
     colsample_bytree=0.9,
     eval_metric='mlogloss'
 )),

 ('lgb', LGBMClassifier(
     n_estimators=400
 )),

 ('svm', SVC(
     probability=True,
     C=15,
     kernel='rbf'
 ))
]

stack_model = StackingClassifier(

    estimators=estimators,

    final_estimator=LogisticRegression()
)


stack_model.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000057 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1588
[LightGBM] [Info] Number of data points in the train set: 320, number of used features: 15
[LightGBM] [Info] Start training from score -0.693147
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fu

C:\Users\USER\miniconda3\envs\py_gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\USER\miniconda3\envs\py_gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\USER\miniconda3\envs\py_gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

C:\Users\USER\miniconda3\envs\py_gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\USER\miniconda3\envs\py_gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,estimators,"[('rf', ...), ('xgb', ...), ...]"
,final_estimator,LogisticRegression()
,cv,None
,stack_method,'auto'
,n_jobs,None
,passthrough,False
,verbose,0
,n_estimators,500
,criterion,'gini'
,max_depth,None
,min_samples_split,2


In [14]:
from sklearn.metrics import accuracy_score, classification_report

pred = stack_model.predict(X_test)

print("Accuracy:",accuracy_score(y_test,pred))

print(classification_report(y_test,pred))

Accuracy: 0.975
              precision    recall  f1-score   support

           0       1.00      0.97      0.99        40
           1       1.00      0.95      0.97        20
           2       0.91      1.00      0.95        20

    accuracy                           0.97        80
   macro avg       0.97      0.97      0.97        80
weighted avg       0.98      0.97      0.98        80



C:\Users\USER\miniconda3\envs\py_gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
